# Lab 2.1. Extraction and validation: from a project annex to a table

**Module 2, Session 2. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

By the end of this notebook you should be able to:

1. Tell a native PDF from a scanned one, and choose the extraction route accordingly.
2. Extract a table that spans several pages and rebuild it without duplicating the header.
3. Validate the extracted table against statements in the document itself and against physical ranges.
4. Use an AI assistant to extract a table you cannot extract by rules, and verify its output with a five-line log.
5. Record the provenance of an extracted value so that someone else can go back to the original.

**How to work.** Statement, code gap, check cell with `assert`. Exercises marked *extension* are
optional. **Predict before you run** the cells marked as such.

**Before handing in:** Kernel, Restart & Run All.

*Note on names.* The documents are Spanish, so their column headers (`Sondeo`, `Prof. (m)` and so on) are
kept as they arrive; variables and new columns are in English.

## 0. Setup

This cell locates the course folder wherever the notebook is running: on your own machine, in
Colab from the course repository, or in Colab from the shared Drive folder. Run it first and
check that the file listing appears.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                             # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):                   # local clone
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")
    root = Path("/content/acs-mod2")               # 1. try the repository
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root
    from google.colab import drive                     # 2. fall back to Drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError(
        "Could not find the course folder. Either set REPO to the course repository, "
        f"or add a shortcut to the shared folder in My Drive as {DRIVE}.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
print("Course folder:", BASE)
print("Working folder:", WORK)
sorted(p.name for p in DATA.iterdir())

### Installs (Colab only)

Locally these are already installed. In Colab the cell installs `pdfplumber` and the PDF tools;
Tesseract with Spanish language data is only needed for the optional OCR extension and takes
about a minute, so it is left commented out. Uncomment it at the start of the break if you plan
to do extension B.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

ensure("pdfplumber")
ensure("pyarrow")
if Path("/content").exists():                          # Colab: poppler for pdffonts / pdftoppm
    subprocess.run(["apt-get", "install", "-y", "-q", "poppler-utils"],
                   check=False, capture_output=True)
    # For extension B (OCR), uncomment the two lines below:
    # subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr", "tesseract-ocr-spa"], check=False)
    # ensure("pytesseract")
print("ready")

In [ ]:
import pandas as pd, numpy as np, matplotlib
import matplotlib.pyplot as plt
import pdfplumber, subprocess
for m in (pd, np, matplotlib, pdfplumber):
    print(f"{m.__name__:<11}", m.__version__)

## Part 1. The report, by rules

### Before extracting: native or scanned?

A PDF is not a data format: it is a page-description language. It stores instructions to draw
pieces of text at coordinates. It does not know what a paragraph is, nor a table; the tool reconstructs
that from positions and drawn lines.

The first decision is whether the file has a text layer. A **native** PDF has one; a **scanned**
PDF is one image per page, whatever the extension says. The test takes two seconds: if `pdffonts`
lists no font, it is an image.

In [ ]:
for f in ["anejo_geotecnico.pdf", "certificado_hormigon_escaneado.pdf"]:
    out = subprocess.run(["pdffonts", str(DATA / f)], capture_output=True, text=True).stdout
    fonts = [l for l in out.split("\n")[2:] if l.strip()]
    print(f"{f:<40} {len(fonts)} fonts -> {'NATIVE' if fonts else 'SCANNED'}")

Two documents, two routes. The geotechnical annex is native: rules (`pdfplumber`) will do.
The concrete test certificate is a scan: rules see nothing, OCR replaces characters without
warning, and an AI assistant will return a beautiful table that may be wrong. Exercises 1 to 5
take the first route; exercise 6 takes the third, under a protocol.

### Exercise 1. Locate

A real annex has hundreds of pages. Finding the table by hand does not scale.
Walk the pages of the annex with `pdfplumber`, look for the word `SPT` in the text, and store in
`table_pages` the list of 0-based page indices that also contain at least one detected table.

In [ ]:
annex_path = DATA / "anejo_geotecnico.pdf"

# YOUR CODE HERE
raise NotImplementedError
print("Pages with an SPT table:", table_pages)

In [ ]:
assert table_pages == [1, 2], f"Expected pages 1 and 2, got {table_pages}"
print("Checks passed: the table spans two pages.")

> A table spanning two pages is not a quirk of this example: it is the normal case in an annex.
> Extracting only the first page is the most frequent error and the hardest to detect, because
> the result looks complete.

### Exercise 2. Extract

Extract the table from each of the two pages and look at the raw result before turning it into
anything. Store the two lists of lists in `t1` and `t2`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print("First page of the table :", len(t1), "rows")
print("Second page of the table:", len(t2), "rows")
print("\nFirst row of each:")
print(t1[0])
print(t2[0])

In [ ]:
assert len(t1) == 43 and len(t2) == 22, f"Expected 43 and 22 rows, got {len(t1)} and {len(t2)}"
assert t1[0] == t2[0], "Both pages should start with the same header"
print("Checks passed: the header is repeated on the second page.")

### Exercise 3. Reconstruct

**Predict before you run:** if you simply concatenate `t1` and `t2`, how many rows will you get,
and how many should there be?

Join the two parts into a single `DataFrame` called `spt`, using the first row as header and
**without** carrying the repeated header of the second page. Then check the row count against
what the annex text says: section 1 states that 63 SPT tests were carried out.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(spt.shape)
spt.head(3)

In [ ]:
assert len(spt) == 63, f"The annex says 63 tests and you have {len(spt)}"
assert not (spt.iloc[:, 0] == "Sondeo").any(), "The repeated header slipped in as a data row"
assert spt["Sondeo"].nunique() == 6, "The annex says 6 boreholes"
print("Checks passed: 63 tests in 6 boreholes, as the text says.")

### Exercise 4. Type

Everything that comes out of a PDF is text. Convert the numeric columns to numbers, remembering
that the annex uses a decimal comma, and rename the columns so that they carry their units:
`borehole`, `depth_m`, `unit`, `n_spt`, `moisture_pct`, `density_g_cm3`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(spt.dtypes)
spt.head(3)

In [ ]:
assert spt["depth_m"].dtype.kind == "f"
assert str(spt["n_spt"].dtype) in ("Int64", "int64")
assert spt[["depth_m", "n_spt", "moisture_pct"]].notna().all().all(), (
    "Some conversion produced NaN: check the decimal comma")
print("Checks passed.")

### Exercise 5. Validate

The step almost everyone skips. Validating is not looking whether the table "seems fine": it is
checking concrete statements, against the document itself and against the physics of the problem.

Write the five checks as `assert`:

1. The number of tests matches the one the text declares (63).
2. The number of boreholes matches (6).
3. Within each borehole, depth is strictly increasing.
4. The blow count N lies between 0 and 100, the physical range of the test without refusal.
5. Moisture lies between 0 and 100 per cent, and bulk density between 1.0 and 2.5 g/cm³.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print("All five checks pass.")
print(spt[["depth_m", "n_spt", "moisture_pct", "density_g_cm3"]].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 6))
for b, g in spt.groupby("borehole"):
    ax.plot(g["n_spt"], g["depth_m"], "o-", label=b, markersize=4, linewidth=1)
ax.invert_yaxis()
ax.set_xlabel("N (SPT), blows"); ax.set_ylabel("depth (m)")
ax.set_title("Blow count versus depth, by borehole")
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout()

The plot is the sixth check, and the cheapest: a wrongly extracted value almost always shows as a
point outside the cloud. If any series crossed or went backwards, you would go back to the PDF.

## Part 2. The certificate, by assistant

The concrete test certificate has no text layer. Rules see nothing. Optical character recognition
works (extension B shows it), but it replaces characters without warning. The route most people
take today is an AI assistant, and this exercise teaches the only thing that separates
professional use from careless use: **the verification protocol**.

### Exercise 6. AI extraction, under protocol

1. Open `data/certificado_hormigon_escaneado.pdf` (or render it below) and give the page to any
   assistant you use; the phone is fine.
2. Ask for the table as CSV, semicolon-separated, keeping the Spanish decimal comma.
3. Paste what it returns into the string `pasted_csv` below and load it.
4. Verify it: 12 specimens, strengths in a plausible range, and the ages 7 or 28 days.
5. Fill in the five-line verification log and set `LOG_COMPLETE = True`.

The solution version pastes the ground truth so that the notebook runs end to end. Your own paste
will differ, and **that is the point**: the checks, not the tool, decide whether you may use it.

In [ ]:
subprocess.run(["pdftoppm", "-r", "110", "-png", str(DATA / "certificado_hormigon_escaneado.pdf"),
                str(WORK / "certificate")], check=True)
from IPython.display import Image as IPImage, display
display(IPImage(filename=str(WORK / "certificate-1.png"), width=620))

In [ ]:
import io

pasted_csv = """
# YOUR CODE HERE
raise NotImplementedError
"""

cert = pd.read_csv(io.StringIO(pasted_csv.strip()), sep=";", decimal=",")
cert.columns = [c.strip() for c in cert.columns]
cert

In [ ]:
# Verification: the checks decide, not the tool.
assert len(cert) == 12, f"The certificate says 12 specimens; the assistant returned {len(cert)}"
strength = pd.to_numeric(cert.iloc[:, -1], errors="coerce")
age = pd.to_numeric(cert.iloc[:, 3], errors="coerce")
assert strength.notna().all(), "Some strength value is not a number: a decimal comma probably went missing"
assert strength.between(15, 50).all(), f"Strength outside 15-50 MPa: {strength.min():.1f} to {strength.max():.1f}"
assert set(age.dropna().unique()) <= {7, 28}, f"Ages should be 7 or 28 days, got {sorted(age.unique())}"
print("Range checks passed. Now the spot checks and the log.")

Range checks cannot catch a wrong digit that stays in range. Pick three cells at random and
compare them with the image above, by eye. Then fill in the log.

In [ ]:
verification_log = {
    "source":       "",   # document, page, edition
    "tool_and_date": "",  # which assistant, when
    "totals":       "",   # recomputed vs printed, result
    "spot_checks":  "",   # 3 random cells vs the page, result
    "counts_units": "",   # rows, columns, units, discrepancies
}
LOG_COMPLETE = False
# YOUR CODE HERE
raise NotImplementedError
for k, v in verification_log.items():
    print(f"{k:<14} {v}")

In [ ]:
assert all(v.strip() for v in verification_log.values()), "Every line of the log must be filled in"
assert LOG_COMPLETE, "Set LOG_COMPLETE = True once the three spot checks are done"
print("Checks passed: the table is verified AND the log is written. Only now may it enter an analysis.")

> **What the log is for.** Six months from now nobody will remember which assistant produced this
> table or which cells were checked. The log is part of the deliverable whenever AI-extracted data
> enters an analysis. And before either route, ask whether the source spreadsheet exists: most
> published PDFs began life as someone's Excel.

### Exercise 7. Provenance

A number extracted without its provenance is unusable six months later, because nobody can go back
to the original to check it.

Add four provenance columns to `spt` and save the result as `spt_validated.parquet`:
the file name, the pages it came from, the extraction date and the tool.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(spt_final.shape)
spt_final.head(2)

In [ ]:
reloaded = pd.read_parquet(WORK / "spt_validated.parquet")
assert {"source_file", "source_pages", "extraction_date", "tool"} <= set(reloaded.columns)
assert len(reloaded) == 63
print("Checks passed. The data are now traceable.")

### Extension A. Read the control figure from the text itself

In exercise 5 you checked against 63, a number you read by hand. That does not scale to a hundred
annexes. Extract it automatically from the first page with a regular expression that looks for a
number followed by the phrase "ensayos de penetración estándar".

In [ ]:
import re
# YOUR CODE HERE
raise NotImplementedError
print("The text declares", declared, "tests; the table has", len(spt))
assert declared == len(spt)

### Extension B. The second route: OCR

Requires Tesseract with Spanish data (uncomment the install lines in the setup cell). Rasterise the
certificate at 300 dpi, **convert to greyscale**, run Tesseract with `lang="spa"` and `--psm 6`,
and compare with the image. Then look at the confidence per word: the mean is above 90 and there
are still wrong cells. High confidence means the engine is sure, not that the value is right.

In [ ]:
try:
    import pytesseract
    from PIL import Image
    subprocess.run(["pdftoppm", "-r", "300", "-png", str(DATA / "certificado_hormigon_escaneado.pdf"),
                    str(WORK / "cert300")], check=True)
    img = Image.open(WORK / "cert300-1.png").convert("L")
    # YOUR CODE HERE
    raise NotImplementedError
    for line in [l for l in ocr_text.split("\n") if l.strip()][:20]:
        print(line)
    print(f"\nWords: {len(data)} | mean confidence: {data['conf'].mean():.1f} | below 80: {(data['conf'] < 80).sum()}")
    print("Look for: the isolated digit 7 read as a letter; a decimal comma dropped; IIa read as lla.")
except ImportError:
    print("pytesseract not installed: uncomment the OCR lines in the setup cell to run this extension.")

### Extension C. A monitoring series

`piezometro_PZ07.csv` is a Campbell datalogger export in TOA5 format: four header lines, units on the
third, data from the fifth. Read it correctly, identify the sentinel value and count the affected
records. Sessions 6 and 7 use this series.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(f"Records: {len(pz)} | sentinels -99.99: {n_sentinel} | duplicated timestamps: {n_dup}")
assert len(pz) > 1300 and n_sentinel > 100 and n_dup == 1
print("Checks passed. The duplicated timestamp is the October clock change.")

---

## Take-aways

- A PDF stores pieces of text at coordinates, not tables. The tool rebuilds the table.
- `pdffonts` tells a native from a scanned PDF in two seconds and decides the whole strategy.
- A table spread over pages is the normal case; extracting only the first page looks complete.
- Validating means checking concrete statements: count against the text, monotonicity, physical range.
- Rules fail visibly; OCR fails silently; assistants fail convincingly. The protocol is the same for all three.
- Without provenance, and without the log, an extracted value cannot be checked and therefore cannot be used.

**References.** Lau, Gonzalez and Nolan (2023), ch. 13. Rule et al. (2019), rules 2 and 8.